In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import os
import glob
import gzip
import shutil

COUNTRY = "Australia"
YEARS = range(2022, 2025)
UNZIP_DIR = "./openaq_unzipped/Australia"
RAW_DIR="/home/rishi/ML Projects/Air Pollution/openaq_raw2"
def unzip_country(country, years, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    pattern_list = []
    for year in years:
        pattern = os.path.join(RAW_DIR, country, "records", "csv.gz",
                               "locationid=*", f"year={year}", "month=*", "*.csv.gz")
        pattern_list.extend(glob.glob(pattern))

    def _unzip_one(gz_path):
        out_path = os.path.join(out_dir, os.path.basename(gz_path).replace(".gz", ""))
        if os.path.exists(out_path):
            return 0
        with gzip.open(gz_path, "rb") as f_in, open(out_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        return 1

    with ThreadPoolExecutor(max_workers=16) as ex:
        results = list(tqdm(ex.map(_unzip_one, pattern_list), total=len(pattern_list), desc="Unzipping"))
    print(f"Unzipped {sum(results)} new files ({len(pattern_list)} total) to {out_dir}")

unzip_country(COUNTRY, YEARS, UNZIP_DIR)

In [8]:
import re
import pandas as pd
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor

OUTPUT_DIR = "./openaq_processed/Australia"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── pollutant mapping (openaq name → standard column name, file suffix) ──
POL_MAP = {
    "pm25": ("PM2.5 (µg/m³)", "PM2.5"),
    "pm10": ("PM10 (µg/m³)",  "PM10"),
    "no2":  ("NO2 (µg/m³)",   "NO2"),
    "so2":  ("SO2 (µg/m³)",   "SO2"),
    "co":   ("CO (mg/m³)",    "CO"),
    "o3":   ("Ozone (µg/m³)", "Ozone"),
}

# ppm → target unit conversion factors
PPM_FACTOR = {
    "no2": 1.88 * 1000,   # ppm → µg/m³
    "so2": 2.62 * 1000,   # ppm → µg/m³
    "o3":  1.96 * 1000,   # ppm → µg/m³
    "co":  1.15,           # ppm → mg/m³
}
# µg/m³ → target unit (only CO needs conversion)
UGM3_FACTOR = {"co": 1 / 1000}  # µg/m³ → mg/m³

def convert_value(value, parameter, units):
    """Convert a value to standard units based on its original unit."""
    if units == "ppm" and parameter in PPM_FACTOR:
        return value * PPM_FACTOR[parameter]
    if units == "µg/m³" and parameter in UGM3_FACTOR:
        return value * UGM3_FACTOR[parameter]
    return value  # already in correct units

# ── group CSV files by location_id ──
all_csvs = glob.glob(os.path.join(UNZIP_DIR, "location-*.csv"))
loc_files = defaultdict(list)
for f in all_csvs:
    m = re.match(r"location-(\d+)-", os.path.basename(f))
    if m:
        loc_files[m.group(1)].append(f)

print(f"Found {len(all_csvs)} CSVs across {len(loc_files)} locations")

Found 85102 CSVs across 172 locations


In [9]:
def process_location(loc_id, files, output_dir):
    """Read all CSVs for one location, split by pollutant, convert units, save."""
    from datetime import timezone, timedelta
    dfs = []
    for f in files:
        try:
            dfs.append(pd.read_csv(f))
        except Exception:
            continue
    if not dfs:
        return loc_id, 0

    df = pd.concat(dfs, ignore_index=True)

    # keep only known pollutants
    df = df[df["parameter"].isin(POL_MAP)]
    if df.empty:
        return loc_id, 0

    # parse as UTC first (handles mixed offsets from DST), then convert to local
    df["Timestamp"] = pd.to_datetime(df["datetime"], utc=True)
    offsets = df["datetime"].str.extract(r"([+-]\d{2}:\d{2})$")[0]
    most_common_offset = offsets.mode().iloc[0]
    h, m = int(most_common_offset[:3]), int(most_common_offset[0] + most_common_offset[4:])
    local_tz = timezone(timedelta(hours=h, minutes=m))
    df["Timestamp"] = df["Timestamp"].dt.tz_convert(local_tz).dt.tz_localize(None)

    saved = 0
    for param, group in df.groupby("parameter"):
        col_name, suffix = POL_MAP[param]

        g = group[["Timestamp", "value", "units"]].copy()
        g["value"] = g.apply(lambda r: convert_value(r["value"], param, r["units"]), axis=1)
        g = g.rename(columns={"value": col_name})[["Timestamp", col_name]]
        g = g.drop_duplicates(subset="Timestamp").set_index("Timestamp").sort_index()

        out_path = os.path.join(output_dir, f"site_{loc_id}_{suffix}.csv")
        g.to_csv(out_path)
        saved += 1

    return loc_id, saved

# ── run in parallel across locations ──
def process_all(loc_files, output_dir, max_workers=8):
    tasks = [(lid, files, output_dir) for lid, files in loc_files.items()]

    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(process_location, *t): t[0] for t in tasks}
        results = []
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Processing locations"):
            try:
                results.append(fut.result())
            except Exception as e:
                print(f"Failed on location {futures[fut]}: {e}")

    total_files = sum(r[1] for r in results)
    print(f"Saved {total_files} pollutant files across {len(results)} locations to {output_dir}")

process_all(loc_files, OUTPUT_DIR)

Processing locations: 100%|██████████| 172/172 [00:10<00:00, 15.98it/s]

Saved 471 pollutant files across 172 locations to ./openaq_processed/Australia
